# 03. Bulk structures and the Materials Project

- Teaches where the bulk crystal at the start of every surface calculation comes from: a file on disk, or the Materials Project database
- A bulk is the material as an endless crystal, with no surface; module 04 cuts every slab of this course out of one
- Before you start: modules 00, 01 and 02 finished, the `mlip-mace` environment active, and Jupyter Lab running
    - The kernel shown at the top right of this notebook is "Python 3 (ipykernel)", the Python of that environment
    - A Materials Project key is optional: without one the notebook still runs from top to bottom, on the files already in `structures/`
- Which bulk a project starts from is canon `B1`, a decision your supervisor records in the project's `CANON.md`, and never one a script makes for you
    - `B1` asks three things: download from a database or build from a prototype, which database entry, and by which rule that entry was picked
- After this you can read a bulk file and its symmetry, choose between its primitive and its conventional cell, and fetch a new one from the Materials Project with your own key

## 0. The setup cell

- Run this cell first, every time you open the notebook
- This module uses two structure packages: ASE, from module 02, and pymatgen, introduced in the section on loading a bulk from a file
    - Each section below imports the part it needs, so you can see where every name comes from
    - The sections build on each other, so run them from top to bottom the first time
- `from pathlib import Path` takes the `Path` tool out of `pathlib`, which comes with Python
- `numpy` is imported here under the short name `np`, and `pandas` under `pd`, as in modules 01 and 02
    - NumPy is used once, to compare two sets of atom positions; pandas prints every table of this notebook
- `WORK` is the folder this notebook writes its own files into
    - `WORK.mkdir(exist_ok=True)` creates the folder, and does nothing if it is already there
    - The folder sits inside this module folder and is not saved to the repository
- `STRUCTURES` points at the shared structure files, two folder levels up
    - `STRUCTURES.exists()` answers `True` or `False`, and is worth asking before reading from it
    - A relative path such as `work` or `../../structures` is read from the folder this notebook file sits in, whatever folder Jupyter Lab was started from

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

WORK = Path("work")
WORK.mkdir(exist_ok=True)
STRUCTURES = Path("../../structures")

print("Folder this notebook writes into")
print(WORK)
print("Folder the starting structures come from")
print(STRUCTURES)
print("Does the structures folder exist")
print(STRUCTURES.exists())

Folder this notebook writes into
work
Folder the starting structures come from
../../structures
Does the structures folder exist
True


## 1. Load a bulk from a file

- A bulk is the material as an endless crystal, with no surface: atoms in a box that repeats in all three directions
    - A surface calculation starts from a bulk, because the slab of module 04 is a piece cut out of it
- A unit cell is that repeating box: three cell vectors, plus the positions of the atoms inside them
- The lattice parameters are the six numbers that describe the box: the three cell vector lengths `a`, `b` and `c` in Å, and the three angles between them `alpha`, `beta` and `gamma` in degrees
- A space group is the complete list of symmetry operations a crystal has, named by a symbol and by a number between 1 and 230
    - `Fd-3m` (#227) is the space group of the normal spinel, the arrangement the Co3O4 of this course takes
    - The symbol is the first thing to read off a structure file: it says which phase of the material the file holds
- pymatgen, the Python Materials Genomics package, is the second structure package of this course
    - ASE holds structures and talks to calculators; pymatgen holds structures and knows crystallography, which is what this module needs
    - Module 00 installed both into the `mlip-mace` environment
- Reading a file with pymatgen
    - `from pymatgen.core import Structure` brings in the pymatgen structure class, the counterpart of the ASE `Atoms` object of module 02
    - `Structure.from_file(path)` reads a structure file and gives a `Structure` back, guessing the format from the file name
    - `structure.composition.reduced_formula` is the formula in its smallest whole-number form, `Co3O4` and not `Co24O32`
    - `structure.num_sites` is the number of atoms, the counterpart of `len(atoms)` in ASE
    - `structure.lattice.a`, `.b` and `.c` are the cell vector lengths in Å, `.alpha`, `.beta` and `.gamma` the angles in degrees, and `.volume` the volume in Å³
    - `structure.cart_coords` is the atom positions in Å, a NumPy array with one row per atom and three columns
- Reading the symmetry
    - `from pymatgen.symmetry.analyzer import SpacegroupAnalyzer` brings in the tool that works the symmetry out of the atom positions
    - `SpacegroupAnalyzer(structure)` looks at the structure once, and `.get_space_group_symbol()` and `.get_space_group_number()` then report the symbol and the number
    - It measures the symmetry instead of reading it from the file, so it also works on a file that carries none, such as a `POSCAR`
- The two file formats of this section
    - `POSCAR` is the VASP format of module 02; the crystal files in `structures/` are written in it and carry the ending `.vasp`
    - `.cif` is the Crystallographic Information File, the format structure databases and viewers exchange
    - `structures/` holds no `.cif` file, so the cell below writes one into `work/` from the `POSCAR` and reads it back, which shows that the two formats carry the same crystal
    - `structure.to(filename=..., fmt="cif")` writes a structure to a file, and `fmt="poscar"` writes the other format
- Moving a structure between the two packages
    - `from pymatgen.io.ase import AseAtomsAdaptor` brings in the translator between them
    - `AseAtomsAdaptor.get_atoms(structure)` gives an ASE `Atoms` object back, and `AseAtomsAdaptor.get_structure(atoms)` goes the other way
    - The translation carries the cell, the positions and the chemical species; it carries no calculator and no constraint
- pandas prints every table of this notebook, as in module 01
    - `pd.DataFrame(rows)` builds a table out of a list of dictionaries: each key becomes a column
    - `.to_string(index=False)` prints it as aligned text and leaves the row numbers out
- NumPy checks the round trip through ASE
    - `np.abs(values)` replaces every number of an array by its absolute value, and `.max()` is the largest of them
    - Zero means the two sets of positions are identical, so nothing was lost on the way to ASE and back
- In the table:
    - Two rows, one crystal: the same formula, atom count, space group and lattice parameters read out of the two formats
    - The file in `structures/` holds the conventional cell of the Materials Project entry `mp-18748`, 56 atoms, cubic with a = 8.0459 Å

In [2]:
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

poscar_path = STRUCTURES / "Co3O4_mp-18748.vasp"
cif_path = WORK / "Co3O4_mp-18748.cif"

from_poscar = Structure.from_file(poscar_path)
from_poscar.to(filename=str(cif_path), fmt="cif")
from_cif = Structure.from_file(cif_path)

format_rows = []
for source, structure in [("POSCAR", from_poscar), ("CIF", from_cif)]:
    analyzer = SpacegroupAnalyzer(structure)
    format_rows.append({"read from": source,
                        "formula": structure.composition.reduced_formula,
                        "atoms": structure.num_sites,
                        "space group": analyzer.get_space_group_symbol(),
                        "number": analyzer.get_space_group_number(),
                        "a (Å)": round(structure.lattice.a, 4),
                        "b (Å)": round(structure.lattice.b, 4),
                        "c (Å)": round(structure.lattice.c, 4),
                        "alpha (°)": round(structure.lattice.alpha, 2),
                        "beta (°)": round(structure.lattice.beta, 2),
                        "gamma (°)": round(structure.lattice.gamma, 2)})

print("File the structure was read from")
print(poscar_path)
print("File written in the CIF format and read back")
print(cif_path)

print()
print("The same crystal read from the two formats")
print(pd.DataFrame(format_rows).to_string(index=False))

atoms = AseAtomsAdaptor.get_atoms(from_poscar)
round_trip = AseAtomsAdaptor.get_structure(atoms)

print()
print("Atoms in the ASE object made from the pymatgen structure")
print(len(atoms))
print("Chemical formula ASE reports for it")
print(atoms.get_chemical_formula())
print("Atoms in the pymatgen structure made back from the ASE object")
print(round_trip.num_sites)
print("Largest position difference over the round trip (Å)")
print(round(float(np.abs(round_trip.cart_coords - from_poscar.cart_coords).max()), 6))

File the structure was read from
../../structures/Co3O4_mp-18748.vasp
File written in the CIF format and read back
work/Co3O4_mp-18748.cif

The same crystal read from the two formats
read from formula  atoms space group  number  a (Å)  b (Å)  c (Å)  alpha (°)  beta (°)  gamma (°)
   POSCAR   Co3O4     56       Fd-3m     227 8.0459 8.0459 8.0459       90.0      90.0       90.0
      CIF   Co3O4     56       Fd-3m     227 8.0459 8.0459 8.0459       90.0      90.0       90.0

Atoms in the ASE object made from the pymatgen structure
56
Chemical formula ASE reports for it
Co24O32
Atoms in the pymatgen structure made back from the ASE object
56
Largest position difference over the round trip (Å)
0.0


## 2. Primitive and conventional cells

- The same crystal can be written in boxes of different size, and the choice sets the number of atoms, and with it the cost of every calculation
- The primitive cell is the smallest box that reproduces the whole crystal by repetition
    - For Co3O4 it holds 14 atoms, in a rhombohedral box: three equal lengths of 5.6893 Å with three angles of 60°
    - Rhombohedral means a box whose three edges are equally long and whose three angles are equal but not 90°
- The conventional cell is the larger box that shows the symmetry of the crystal, the one crystallography tables and structure databases quote
    - For Co3O4 it holds 56 atoms, four times the primitive cell, in the cube of 8.0459 Å
    - Four times the atoms is roughly four times the cost of any calculation, for the same material
- Getting either box out of a structure
    - `analyzer.get_primitive_standard_structure()` gives the primitive cell back
    - `analyzer.get_conventional_standard_structure()` gives the conventional cell back
    - Standard means the shape and the axis order laid down in the international crystallography tables, so that two people who run this get the same box
- Use the conventional cell when a slab is to be cut, which is the case in this course
    - A surface is named by its Miller index, three whole numbers such as (1 0 0) or (1 1 1) that say how the cutting plane crosses the three axes
    - Those three numbers are read against the axes of the conventional cell, because that is the definition crystallography uses
    - Cutting (1 0 0) out of the rhombohedral primitive box gives a different plane of atoms under the same name, so the surface would be the wrong one
    - Module 04 cuts its slabs from the conventional cell, and the capstone relaxes the 56-atom conventional Co3O4 cell directly
- Canon `S7`: the file read here is not relaxed, and a slab is never cut from unrelaxed coordinates
    - The project first relaxes the bulk with its own model, then cuts the slab from the relaxed cell
    - Module 07 is where a relaxation is run, and the capstone is where a bulk cell and its atoms are relaxed together
    - `structures/README.md` records that every file in that folder is a starting point only
- In the table:
    - Three rows, one material: the cell as the file holds it, its primitive cell and its conventional cell
    - The file already holds the conventional cell, so its row and the conventional row agree
    - The conventional cell has four times the atoms and four times the volume of the primitive cell, which is the check that nothing was gained or lost

In [3]:
analyzer = SpacegroupAnalyzer(from_poscar)
primitive = analyzer.get_primitive_standard_structure()
conventional = analyzer.get_conventional_standard_structure()

cell_rows = []
for source, structure in [("as read from the file", from_poscar),
                          ("primitive standard", primitive),
                          ("conventional standard", conventional)]:
    cell_rows.append({"cell": source,
                      "atoms": structure.num_sites,
                      "a (Å)": round(structure.lattice.a, 4),
                      "b (Å)": round(structure.lattice.b, 4),
                      "c (Å)": round(structure.lattice.c, 4),
                      "alpha (°)": round(structure.lattice.alpha, 2),
                      "beta (°)": round(structure.lattice.beta, 2),
                      "gamma (°)": round(structure.lattice.gamma, 2),
                      "volume (Å³)": round(structure.volume, 2)})

print("One Co3O4 crystal in three boxes")
print(pd.DataFrame(cell_rows).to_string(index=False))

print()
print("Atoms in the primitive cell")
print(primitive.num_sites)
print("Atoms in the conventional cell")
print(conventional.num_sites)

print()
print("Space group of the primitive cell")
print(SpacegroupAnalyzer(primitive).get_space_group_symbol())
print("Space group of the conventional cell")
print(SpacegroupAnalyzer(conventional).get_space_group_symbol())

One Co3O4 crystal in three boxes
                 cell  atoms  a (Å)  b (Å)  c (Å)  alpha (°)  beta (°)  gamma (°)  volume (Å³)
as read from the file     56 8.0459 8.0459 8.0459       90.0      90.0       90.0       520.86
   primitive standard     14 5.6893 5.6893 5.6893       60.0      60.0       60.0       130.21
conventional standard     56 8.0459 8.0459 8.0459       90.0      90.0       90.0       520.86

Atoms in the primitive cell
14
Atoms in the conventional cell
56

Space group of the primitive cell
Fd-3m
Space group of the conventional cell
Fd-3m


## 3. Get your own Materials Project key

- The Materials Project is a public database of crystal structures and of properties computed for them, free to use after you register
    - It is where the Co3O4 and CoOOH files in `structures/` came from, and `structures/README.md` records the entry and the date for each
- An API is the door a program uses to ask a database for data, instead of a person clicking through a web page; API stands for application programming interface
- An API key is a long piece of text that identifies you to that database when your program knocks on that door
    - It is a password, not a name: anyone who has it can use your account
- Getting the key, three steps in a web browser
    1. Open `https://next-gen.materialsproject.org` and make a free account
    2. Sign in, then open `https://next-gen.materialsproject.org/api`
    3. Copy the key shown on that page, a string of about 32 characters
- An environment variable is a named piece of text that the operating system hands to every program you start
    - `MP_API_KEY` is the name that every tool in this course looks for, and the name the Materials Project software expects
    - Storing the key there keeps it out of every file you write, so it cannot travel with a notebook or a script
- Store it in your shell profile, so that it is there again every time you open a terminal
    - macOS, zsh: add the line `export MP_API_KEY="your_key_here"` to the file `~/.zshrc`
    - Linux, bash: add the same line to the file `~/.bashrc`
    - Windows, PowerShell: add the line `$Env:MP_API_KEY = "your_key_here"` to the file whose path the command `$PROFILE` prints; the folder above that file may have to be created first
    - Windows, without PowerShell: open the Start menu, type "Edit the system environment variables", press Environment Variables, press New, and give the name `MP_API_KEY` and the key as the value
- The word `export` is what matters on macOS and Linux
    - A variable that is set but not exported stays inside the shell and is invisible to Python
    - This is the most common reason for a key that is "set" and still not found
- Close the terminal and open a new one, then stop Jupyter Lab and start it again
    - A program reads the environment variables once, when it starts, so a Jupyter Lab that was already running will not see the new key
- Check that it worked, on your own screen, before you go on
    - macOS and Linux: `echo $MP_API_KEY` in a terminal prints the key
    - Windows PowerShell: `echo $Env:MP_API_KEY` prints it
    - Nothing printed means the profile was not reloaded, or the line went into a file your shell does not read
- Never paste the key into a notebook, a script or a message
    - A notebook keeps its output, so a key printed once travels with the file to everyone you send it to
    - The cells below read the key out of the environment variable and never print it
    - If a key does reach a file you shared, go back to the API page and ask for a new one; the old one then stops working

## 4. Query the database

- This section asks the database for every entry with the formula Co3O4, and works without a key as well
- Reading the key out of the environment
    - `os` comes with Python, and `os.environ` is the set of environment variables this program was started with
    - `os.environ.get("MP_API_KEY")` gives the key back, or `None` when the variable is not set
    - `HAVE_KEY` is `True` or `False`, and the cells below check it, so the notebook runs either way
    - Without a key the notebook falls back to `structures/Co3O4_mp-18748.vasp`, the file this repository already holds
- Asking the database, when there is a key
    - `from mp_api.client import MPRester` brings in the client, the object that talks to the Materials Project
    - `MPRester(key, mute_progress_bars=True)` opens the connection; the second argument keeps a progress bar out of the notebook output
    - `with MPRester(...) as mpr:` closes the connection again when the block ends, whatever happens inside it
- Hiding one warning, on purpose
    - `warnings` comes with Python, and `with warnings.catch_warnings():` makes every warning setting inside that block temporary
    - `warnings.simplefilter("ignore")` inside the block hides the warnings raised there, and nowhere else in the notebook
    - The client pulls in a progress-bar package, which warns that it cannot draw an animated bar in this notebook
    - The bar was switched off anyway, so that warning says nothing and is hidden rather than left in the output
    - Hiding a warning is a deliberate act, worth doing only once you have read it and know it does not apply
    - `mpr.materials.summary.search(formula="Co3O4", fields=[...])` asks for every entry with that formula and gives a list of documents back
    - `fields` names the pieces you want back; asking for fewer makes the answer smaller and the query faster
- What each field holds
    - `material_id` is the name of the entry in the database, such as `mp-18748`, and it is what you write down in your notes
    - `formula_pretty` is the formula in its smallest whole-number form
    - `symmetry` holds `.symbol` and `.number`, the space group of the section on loading a bulk from a file
    - `nsites` is the number of atoms in the cell the database serves
    - `structure` is the structure itself, as a pymatgen `Structure`
    - `energy_above_hull` and `formation_energy_per_atom` are the two energies below
- A polymorph is a genuinely different arrangement of the same atoms: same formula, different crystal structure, different energy
    - Diamond and graphite are the textbook pair for carbon, rutile and anatase the pair for TiO2
    - The database returns one row per entry, and two entries with the same formula are not automatically two polymorphs
    - They can be the same structure computed under different assumptions, which is what the four Co3O4 rows turn out to be
- Energy above hull, in two bullets
    - The convex hull is the lowest energy a given set of elements can reach at each composition, counting the option of splitting into two or more other compounds
    - The energy above hull of an entry is how far it sits above that lowest line, in eV per atom
        - 0 means nothing else is cheaper at that composition
        - A positive value means the compound could lower its energy by turning into something else
- Formation energy per atom is a different quantity: the energy change when the compound is made from the pure elements, in eV per atom, negative for a compound that is bound
    - It says the compound is worth making at all; the energy above hull says whether something else at the same composition would be better
    - Below about 0.025 eV/atom is the usual working threshold for "a candidate worth calculating", and it is the default of the group's fetch script, not a law of nature
- Where the numbers in the table come from
    - Every energy in it was computed with density functional theory, the PBE functional, and a Hubbard U correction on the transition-metal oxides
    - They are energies at 0 K with no entropy term, so they say nothing directly about what is stable at the temperature of an experiment
    - A gap of about 0.01 eV/atom between two entries is inside the error of that method, so it orders them on paper and not in the laboratory
- Sorting and reading the table with pandas
    - `.sort_values("E above hull (eV/atom)")` puts the rows in order of that column, smallest first
    - `.iloc[0]` reads the first row of a table, whatever its row number is, so the entry with the lowest energy above hull comes out of it
- Writing the chosen entry out
    - `structure.to(filename=..., fmt="poscar")` writes it into `work/` in the VASP format
    - That file is the saved result of this query, kept so that you can look at what the database served and record where it came from
    - It is not an input to any later module: module 04 cuts its slabs from `structures/Co3O4_mp-18748.vasp`, the file this repository already holds
- In the table:
    - One row per Co3O4 entry the database holds, lowest energy above hull first
    - Read on 2026-09-17 the lowest row was `mp-1271793`, space group `R-3m`, at 0.0000 eV/atom, and `mp-18748`, the cubic `Fd-3m` spinel this course uses, sat about 0.010 eV/atom above it
    - Two different space-group symbols, and still the same spinel framework underneath; the section on what to check before using a bulk shows how that was verified
    - The `space group` column is the symmetry the database recorded for the entry, not one measured here
    - pymatgen measuring the symmetry again from the served positions can report a lower one, because the two allow an atom a different distance from its ideal position
    - The database is recomputed and extended over time, so your own numbers may differ from those; write down the date with the material id

In [4]:
import os

key = os.environ.get("MP_API_KEY")
HAVE_KEY = bool(key)

print("Materials Project key")
if HAVE_KEY:
    print("found in the environment variable MP_API_KEY")
else:
    print("not found, using the cached files in structures/")

Materials Project key
found in the environment variable MP_API_KEY


In [5]:
summary_fields = ["material_id", "formula_pretty", "symmetry", "energy_above_hull",
                  "formation_energy_per_atom", "nsites", "structure"]

if HAVE_KEY:
    import warnings

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        from mp_api.client import MPRester

    with MPRester(key, mute_progress_bars=True) as mpr:
        docs = mpr.materials.summary.search(formula="Co3O4", fields=summary_fields)

    structures_by_id = {}
    entry_rows = []
    for doc in docs:
        material_id = str(doc.material_id)
        structures_by_id[material_id] = doc.structure
        entry_rows.append({"material id": material_id,
                           "formula": doc.formula_pretty,
                           "space group": doc.symmetry.symbol,
                           "number": doc.symmetry.number,
                           "E above hull (eV/atom)": round(doc.energy_above_hull, 4),
                           "formation E (eV/atom)": round(doc.formation_energy_per_atom, 4),
                           "atoms": doc.nsites})

    entry_table = pd.DataFrame(entry_rows).sort_values("E above hull (eV/atom)")
    chosen_id = str(entry_table.iloc[0]["material id"])
    chosen = structures_by_id[chosen_id]

    print("Entries the database returned for the formula Co3O4")
    print(len(docs))
    print()
    print("Every Co3O4 entry, lowest energy above hull first")
    print(entry_table.to_string(index=False))
else:
    chosen_id = "mp-18748"
    chosen = Structure.from_file(STRUCTURES / "Co3O4_mp-18748.vasp")
    cached_analyzer = SpacegroupAnalyzer(chosen)
    entry_table = pd.DataFrame([{"material id": chosen_id,
                                 "formula": chosen.composition.reduced_formula,
                                 "space group": cached_analyzer.get_space_group_symbol(),
                                 "number": cached_analyzer.get_space_group_number(),
                                 "a (Å)": round(chosen.lattice.a, 4),
                                 "b (Å)": round(chosen.lattice.b, 4),
                                 "c (Å)": round(chosen.lattice.c, 4),
                                 "atoms": chosen.num_sites}])

    print("Entries read from the cached file in structures/")
    print(len(entry_table))
    print()
    print("The cached Co3O4 entry, read from the file and not from the database")
    print(entry_table.to_string(index=False))

chosen_path = WORK / ("Co3O4_" + chosen_id + ".vasp")
chosen.to(filename=str(chosen_path), fmt="poscar")

print()
print("Entry taken forward")
print(chosen_id)
print("Atoms in the cell as it was served")
print(chosen.num_sites)
print("File written")
print(chosen_path)

Entries the database returned for the formula Co3O4
4

Every Co3O4 entry, lowest energy above hull first
material id formula space group  number  E above hull (eV/atom)  formation E (eV/atom)  atoms
 mp-1271793   Co3O4        R-3m     166                  0.0000                -1.2903     14
   mp-18748   Co3O4       Fd-3m     227                  0.0105                -1.2799     14
 mp-1271784   Co3O4        Imma      74                  0.0304                -1.2600     14
 mp-1271940   Co3O4    I4_1/amd     141                  0.0899                -1.2004     14

Entry taken forward
mp-1271793
Atoms in the cell as it was served
14
File written
work/Co3O4_mp-1271793.vasp


## 5. The group's fetch script

- `scripts/fetch_bulk.py` does the whole of the section on querying the database from a terminal, in one line and without a notebook
    - It is the script the group uses, and the one that wrote the Co3O4 and CoOOH files in `structures/`
    - It reads the same environment variable `MP_API_KEY`, and stops with an error message when it finds none
- The two commands this section runs, as you would type them in a terminal
    - `python ../../scripts/fetch_bulk.py --formula Pt --output work/Pt_mp.vasp`
    - `python ../../scripts/fetch_bulk.py --formula Pt --list-polymorphs`
    - Type them in this module folder, with `(mlip-mace)` shown in the prompt
- The flags worth knowing
    - `--formula Pt` asks by chemical formula, and `--mp-id mp-126` asks for one entry by its material id
    - `--output work/Pt_mp.vasp` is the file to write; `--format` chooses `vasp`, `cif`, `xyz` or `json`, and `vasp` is the default
    - `--conventional` converts to the conventional cell before writing, which is how the 56-atom Co3O4 file in `structures/` was made
    - `--max-ehull 0.025` is the default ceiling on the energy above hull, so a plain `--formula` query returns near-stable entries only
    - `--list-polymorphs` prints every entry up to 1.0 eV/atom above the hull and writes no file, which is the command to run before you choose
- Running a terminal command from Python, as in module 02
    - `subprocess` comes with Python and starts another program from inside Python
    - `subprocess.run(command, capture_output=True, text=True)` runs it and keeps its output as text instead of printing it
    - `result.stdout` is the normal output, `result.stderr` the error output, and `result.returncode` is 0 when the command worked
    - `sys` also comes with Python, and `sys.executable` is the Python running this notebook, so the script runs in this environment
    - The printed command line says `python`, which is what you type in the terminal
- Without a key the cell prints each command and runs neither, so the notebook still finishes
- In the two reports:
    - The structure summary names the material id, the space group, the lattice parameters and the energy above hull of the entry it saved
    - The script prints `Volume (ų)` where it means `Volume (Å³)`, a text-encoding slip in the script itself; the fix belongs in the copy this file came from, not here
    - The saved `work/Pt_mp.vasp` holds the 1-atom rhombohedral primitive cell the database serves, 2.7882 Å with angles of 60°, and not the 4-atom cube
    - `--conventional` is the flag to add before a slab is to be cut from it, for the reason the section on primitive and conventional cells gives
    - The second report is the same information for every Pt entry, and `mp-126`, face-centred cubic `Fm-3m`, is the entry on the hull
    - This course does not use that file: it builds its Pt with ASE at a lattice constant of 3.92 Å, as `structures/README.md` and `scripts/onboarding_structures.py` record
    - The conventional cell of `mp-126` measures 3.9432 Å, which is a different starting point, and choosing between the two is canon `B1`

In [6]:
import subprocess
import sys

FETCH = Path("../../scripts/fetch_bulk.py")


def run_fetch(arguments):
    """Run the group's fetch script and give its normal output back as text."""
    print("Command")
    print(" ".join(["python", str(FETCH)] + arguments))
    if not HAVE_KEY:
        print("Not run")
        print("no Materials Project key was found")
        return ""
    result = subprocess.run([sys.executable, str(FETCH)] + arguments,
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr.strip()[-500:])
        raise SystemExit("the fetch script failed")
    return result.stdout


saved = run_fetch(["--formula", "Pt", "--output", "work/Pt_mp.vasp"])
if saved:
    print()
    print("What the script reported")
    print(saved.strip())

print()
listed = run_fetch(["--formula", "Pt", "--list-polymorphs"])
if listed:
    print()
    print("Entries for Pt the script listed")
    print(listed.strip())

Command
python ../../scripts/fetch_bulk.py --formula Pt --output work/Pt_mp.vasp



What the script reported
--- Structure Summary ---
Material ID:    mp-126
Formula:        Pt
Space group:    Fm-3m (#225)
Lattice (Å):    a=2.7882  b=2.7882  c=2.7882
Angles (°):     α=60.00  β=60.00  γ=60.00
Volume (ų):   15.33
Num atoms:      1
Ehull (eV/at):  0.0000
Form E (eV/at): 0.0000
Band gap (eV):  0.000
------------------------

Saved to: work/Pt_mp.vasp

Command
python ../../scripts/fetch_bulk.py --formula Pt --list-polymorphs



Entries for Pt the script listed
Polymorphs for Pt (3 found):
Material ID     Space Group          Ehull (eV/at)   Sites    Form. E (eV/at)
---------------------------------------------------------------------------
mp-126          Fm-3m                0.0000          1        0.0000         
mp-2647022      P6_3/mmc             0.0691          2        0.0691         
mp-2646979      Im-3m                0.1070          1        0.1070


## 6. What to check before using a bulk

- Does the space group match the phase you mean
    - The database ranks entries by computed energy, and the lowest one is not always the phase the experiment makes
    - Check the symbol against the phase your experimental colleagues report, and against the reference in the paper you are following
- A different space-group symbol does not by itself mean a different phase, and the Co3O4 of this notebook is the example
    - The lowest entry, `mp-1271793`, is recorded as `R-3m`, while `mp-18748`, the spinel this course uses, is `Fd-3m`
    - `StructureMatcher(ltol=0.3, stol=0.4, angle_tol=8).fit(...)`, from `pymatgen.analysis.structure_matcher`, answers `True` for those two
    - The root-mean-square distance between the matched atoms is 0.028 Å, which is far smaller than any bond in the material
    - `SpacegroupAnalyzer` on the served `mp-1271793` positions reports `C2/m` at a tolerance of 0.01 Å, `R-3m` at 0.1 Å and `Fd-3m` at 0.3 Å
    - The symbol therefore depends on how far an atom is allowed to sit from its ideal position, and is not a fixed property of the file
    - Its cell is 5.88, 5.94 and 5.88 Å at 60.1, 59.4 and 60.1°, against 5.689 Å at 60° for the spinel primitive cell
    - That is 3 to 4 % longer in each direction, and slightly more in one direction than in the other two
    - So it is the same spinel framework, expanded and distorted a little, and given a lower symmetry symbol because of that distortion
    - The most likely reason is a different magnetic ordering used in that calculation, which is an inference your supervisor should confirm and not something this notebook establishes
    - The point for you is that a different space-group symbol in the database does not by itself mean a different phase
    - That is one more reason to ask rather than infer, and canon `B1` is where the answer is written down
- Energy above hull, read as a warning light and not as a verdict
    - 0 eV/atom means nothing else is cheaper at that composition
    - A metastable polymorph, above the hull, can still be the right structure when it is the one the experiment makes and keeps
    - A large value on the entry you intend to use is a reason to stop and ask, not a reason to continue quietly
- Magnetic ordering
    - A magnetic ordering is the arrangement of the electron spins on the magnetic atoms: all parallel, alternating, or something else
    - Co3O4 is antiferromagnetic, so the arrangement changes the energy and can change the relaxed geometry
    - The database entry carries one arrangement, chosen by whoever computed it, and it may not be the one your project wants
    - Which ordering to use is your supervisor's decision, recorded in the project's `CANON.md`, never a default taken from a tool
- Which cell you take forward
    - The database serves the primitive cell unless you ask for the conventional one, and the two differ by a factor of four for Co3O4
    - The section on primitive and conventional cells says which one a slab is cut from
- Canon `B1`, the source of the bulk, is a project decision
    - It covers the source, the entry, and the rule by which the entry was chosen
    - It is recorded in the project's `CANON.md` before the first calculation, and it is not settled by this notebook or by any script
- Canon `S7`, the file is a starting point and never the final geometry
    - Slabs are cut from a bulk relaxed at the project's own level of theory, never from the downloaded coordinates
    - A relaxation that breaks the assumed symmetry makes the idealized coordinates wrong, and the slab built on them wrong with it
- Write the provenance down next to the file
    - The material id, the date it was fetched, the space group, the atom count and the cell it was saved in
    - `structures/README.md` in this repository is the shape to copy: one table row per file, plus a note on why that entry was chosen

## Check yourself

1. You download an entry for your material and it comes back with 14 atoms and three cell angles of 60°. Which cell
   is that, what do you call to get the cubic one, and how many atoms does that one hold?
2. You added the `export` line to `~/.zshrc`, but `os.environ.get("MP_API_KEY")` still gives `None` in this notebook.
   What are the two things to check, in which order?
3. The entry with the lowest energy above hull for your formula has a different space group from the phase your
   experimental colleagues measured. What do you do, and which canon item covers the decision?

<details>
<summary>Answers</summary>

- The primitive cell; `SpacegroupAnalyzer(structure).get_conventional_standard_structure()` gives the conventional
  cell, which for Co3O4 holds 56 atoms, four times as many, in a cubic box
- First open a new terminal and start Jupyter Lab again from it, because a program reads the environment variables
  once when it starts; then check that the line really begins with `export` and sits in the profile file your shell
  reads, since a variable that is set but not exported is invisible to Python
- Do not take it silently, and do not assume it is a different phase either: compare the two structures with
  `StructureMatcher` before you conclude anything, because a distorted copy of the same framework gets a lower
  symmetry symbol and can still match. Whatever the comparison says, the energy above hull ranks computed
  stability at that composition and says nothing about which phase the experiment made, so raise the disagreement
  with your supervisor; which entry the project uses is canon `B1`, and the answer belongs in the project's
  `CANON.md` together with the material id and the date

</details>